In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

data_dir = Path("../data/TUTORIAL_ROUND_1")
price_files = sorted(data_dir.glob("prices_round_0_day_*.csv"))
frames = [pd.read_csv(f, sep=";") for f in price_files]
prices = pd.concat(frames, ignore_index=True)

print("Products:", prices["product"].unique())
print("Shape:", prices.shape)
prices.head()


Products: ['TOMATOES' 'EMERALDS']
Shape: (40000, 17)


,day,timestamp,product,bid_price_1,bid_volume_1,bid_price_2,bid_volume_2,bid_price_3,bid_volume_3,ask_price_1,ask_volume_1,ask_price_2,ask_volume_2,ask_price_3,ask_volume_3,mid_price,profit_and_loss
0,-1,0,TOMATOES,4999,5,4998,15,NaN,NaN,5013,5,5014,15,NaN,NaN,5006.0,0.0
1,-1,0,EMERALDS,9992,14,9990,29,NaN,NaN,10008,14,10010,29,NaN,NaN,10000.0,0.0
2,-1,100,EMERALDS,9992,11,9990,22,NaN,NaN,10008,11,10010,22,NaN,NaN,10000.0,0.0
3,-1,100,TOMATOES,5000,8,4998,21,NaN,NaN,5013,8,5014,21,NaN,NaN,5006.5,0.0
4,-1,200,EMERALDS,9992,15,9990,20,NaN,NaN,10008,15,10010,20,NaN,NaN,10000.0,0.0


In [ ]:
for product, g in prices.groupby("product"):
    mid = g["mid_price"]
    spread = g["ask_price_1"] - g["bid_price_1"]
    print(f"\n=== {product} ===")
    print(f"  mid  -> mean={mid.mean():.1f}  std={mid.std():.2f}  min={mid.min()}  max={mid.max()}")
    print(f"  spread -> mean={spread.mean():.1f}  min={spread.min()}  max={spread.max()}")



=== EMERALDS ===
  mid  → mean=10000.0  std=0.72  min=9996.0  max=10004.0
  spread → mean=15.7  min=8  max=16

=== TOMATOES ===
  mid  → mean=4992.8  std=19.75  min=4946.5  max=5036.0
  spread → mean=13.0  min=5  max=14


In [3]:
import plotly.express as px

fig = px.line(prices, x="timestamp", y="mid_price", color="product",
              facet_row="product", facet_col="day",
              title="Mid-price over time")
fig.update_yaxes(matches=None)
fig.show()


In [ ]:
# How often do bots trade? (i.e. how often would our passive orders fill?)
trade_files = sorted(data_dir.glob("trades_round_0_day_*.csv"))
trades = pd.concat([pd.read_csv(f, sep=";") for f in trade_files], ignore_index=True)

for product, g in trades.groupby("symbol"):
    print(f"\n=== {product} ===")
    print(f"  total trades: {len(g)},  total qty: {g['quantity'].sum()}")
    print(f"  price mean={g['price'].mean():.1f}  min={g['price'].min()}  max={g['price'].max()}")

# For EMERALDS: every trade at 9992 means a bot was selling -> our 9998 bid would have been hit
# Every trade at 10008 means a bot was buying -> our 10002 ask would have been lifted
em_trades = trades[trades["symbol"] == "EMERALDS"]
sells_to_us = em_trades[em_trades["price"] == 9992]
buys_from_us = em_trades[em_trades["price"] == 10008]
print(f"\nEMERALDS market-making opportunity:")
print(f"  bot sells (we'd buy at 9998): {len(sells_to_us)} trades, {sells_to_us['quantity'].sum()} qty -> profit ~{sells_to_us['quantity'].sum() * 2}")
print(f"  bot buys  (we'd sell at 10002): {len(buys_from_us)} trades, {buys_from_us['quantity'].sum()} qty -> profit ~{buys_from_us['quantity'].sum() * 2}")



=== EMERALDS ===
  total trades: 399,  total qty: 2189
  price mean=9999.8  min=9992.0  max=10008.0

=== TOMATOES ===
  total trades: 820,  total qty: 2853
  price mean=4992.6  min=4943.0  max=5040.0

EMERALDS market-making opportunity:
  bot sells (we'd buy at 9998): 199 trades, 1087 qty → profit ~2174
  bot buys  (we'd sell at 10002): 189 trades, 1048 qty → profit ~2096


## Strategy: simple market-making

1. **EMERALDS**: fair = 10000 (fixed). Post buy at 9998, sell at 10002.
2. **TOMATOES**: fair = (best_bid + best_ask) / 2 (dynamic). Post buy at fair-1, sell at fair+1.

In [7]:
t = prices[prices["product"] == "TOMATOES"].copy()

# Per-day breakdown
for day, g in t.groupby("day"):
    m = g["mid_price"]
    print(f"Day {day}: mean={m.mean():.1f}  median={m.median():.1f}  start={m.iloc[0]}  end={m.iloc[-1]}")

print(f"\nOverall: mean={t['mid_price'].mean():.1f}  median={t['mid_price'].median():.1f}")

# How often does price cross 4980?
below = (t["mid_price"] < 4980).mean() * 100
print(f"\n% ticks mid < 4980: {below:.1f}%")

# Profit if we use 4980 as fair: buy everything below, sell everything above
t["edge_buy"]  = (4980 - t["ask_price_1"]).clip(lower=0)  # profit when ask < 4980
t["edge_sell"] = (t["bid_price_1"] - 4980).clip(lower=0)  # profit when bid > 4980
print(f"\nWith fair=4980:")
print(f"  buy  opportunities: {(t['edge_buy'] > 0).sum()} ticks, avg edge {t['edge_buy'][t['edge_buy']>0].mean():.1f}")
print(f"  sell opportunities: {(t['edge_sell'] > 0).sum()} ticks, avg edge {t['edge_sell'][t['edge_sell']>0].mean():.1f}")


Day -2: mean=5007.9  median=5006.0  start=5000.0  end=5006.5
Day -1: mean=4977.6  median=4980.0  start=5006.0  end=4957.0

Overall: mean=4992.8  median=4995.5

% ticks mid < 4980: 24.9%

With fair=4980:
  buy  opportunities: 4168 ticks, avg edge 11.1
  sell opportunities: 13393 ticks, avg edge 17.8
